# SkinMatch: run the full scrape
Put `scrape_skincare.py` and this notebook in the same folder. Run the cells **in order, one at a time**.
Read each site's Terms of Use before scraping it.

## 1. Install (one time)

In [1]:
!pip install requests beautifulsoup4 lxml pandas

In [2]:
# Only needed for Jenpharm (its ingredients load via JavaScript). Skip if you don't need Jenpharm.
!pip install playwright
!playwright install chromium

## 2. Settings
Delete a brand from the list to skip it. Raise `MAX_PER_BRAND` for more products.

In [3]:
# La Roche-Posay is NOT in the list: its site answers 403 (it blocks automated requests). Respect that.
# The Shopify shops answered 429 (too many requests). That block is temporary: wait a few hours,
# then run the check cell first. Scrape them one at a time, e.g. ORDER = ["BnB Derma"].
ORDER = ["BnB Derma", "SkinFactor", "Mazton", "Accufix",
         "The Ordinary", "CeraVe", "Cetaphil", "Blissful"]
MAX_PER_BRAND = 300
MAX_PAGES_PER_BRAND = 400

## 3. Scrape
Safe to stop (press the stop button) and run again: saved products are skipped.
After scraping new brands, run the **Kaggle merge cell** again, then section 6, so the new products reach `combined_filled.csv`.

In [4]:
# Quick check (about 15 seconds): is each site reachable, and does it allow automated requests?
import importlib, scrape_skincare as s
importlib.reload(s)
s.check_sites(ORDER)

BnB Derma        FAIL https://bnbderma.com/products.json?limit=1  -> status 429: too many requests: wait and retry later
SkinFactor       FAIL https://www.skinfactor.pk/products.json?limit=1  -> status 429: too many requests: wait and retry later
Mazton           FAIL https://shopmazton.com/products.json?limit=1  -> status 429: too many requests: wait and retry later
Accufix          FAIL https://accufixcosmetics.com/products.json?limit=1  -> status 429: too many requests: wait and retry later
    request error after 1 tries: https://theordinary.com/robots.txt
      ConnectionError: HTTPSConnectionPool(host='theordinary.com', port=443): Max retries exceeded with url: /robots.txt (Caused by NameResolutionError("HTTPSConnection(host='theordinary.com', port=443): Failed to resolve 'theordinary.com' ([Errno 11002] getaddrinfo failed)"))
The Ordinary     FAIL https://theordinary.com/robots.txt  -> no connection (internet/DNS problem, or the site is unreachable from your network)
CeraVe     

In [5]:
import pandas as pd
import importlib
import scrape_skincare as s
importlib.reload(s)          # IMPORTANT: loads the newest scrape_skincare.py (otherwise new brands are missing)

missing = [b for b in ORDER if b not in s.SITES]
if missing:
    print("Not in scrape_skincare.SITES (old file in this folder?):", missing)

s.MAX_PRODUCTS_PER_BRAND = MAX_PER_BRAND
s.MAX_PAGES_FETCHED_PER_BRAND = MAX_PAGES_PER_BRAND
s.migrate_csv()

results = []
for brand in ORDER:
    if brand not in s.SITES:
        continue
    try:
        done = s.load_done_urls()
        results.append(s.scrape_brand(brand, s.SITES[brand], done))
    except KeyboardInterrupt:
        print("Stopped. Run this cell again to continue.")
        break
    except Exception as e:
        print(f"!! {brand} failed: {type(e).__name__}: {e}")
        results.append({"brand": brand, "error": str(e)[:100]})

print("\n========== SUMMARY ==========")
for r in results:
    print(r)


=== BnB Derma ===
  reminder: read this site's Terms of Use before scraping (robots.txt is not the whole story)
    429 too many requests: waiting 60s before retrying https://bnbderma.com/products.json?limit=250&page=1
    429 too many requests: waiting 60s before retrying https://bnbderma.com/products.json?limit=250&page=1
  products.json not available -> status 429: too many requests: wait and retry later

=== SkinFactor ===
  reminder: read this site's Terms of Use before scraping (robots.txt is not the whole story)
    429 too many requests: waiting 60s before retrying https://www.skinfactor.pk/products.json?limit=250&page=1
    429 too many requests: waiting 60s before retrying https://www.skinfactor.pk/products.json?limit=250&page=1
  products.json not available -> status 429: too many requests: wait and retry later

=== Mazton ===
  reminder: read this site's Terms of Use before scraping (robots.txt is not the whole story)
    429 too many requests: waiting 60s before retrying 

In [6]:
import pandas as pd

KAGGLE_IN = "data/raw/cosmetics.csv"      # the original Kaggle file, never overwritten
OUT       = "data/combined_products.csv"  # merged output goes to a different file

k = pd.read_csv(KAGGLE_IN)
k.columns = [c.strip().lower() for c in k.columns]
print("Kaggle columns:", k.columns.tolist())   # should include label, brand, name, ingredients, dry, oily ...

def skin_text(r):
    return "|".join(n for n in ["combination", "dry", "normal", "oily", "sensitive"] if r.get(n) == 1)

ext = pd.DataFrame({
    "brand": k["brand"], "name": k["name"], "category": k["label"],
    "ingredients": k["ingredients"].str.lower().str.strip(),
    "skin_types": k.apply(skin_text, axis=1).replace("", pd.NA),
    "price_usd": k["price"], "rating": k.get("rank"),
    "currency": "USD", "source": "kaggle (sephora)"})

# drop placeholder / non-list ingredient text ("No Info", "Visit the boutique...")
ok = ext.ingredients.fillna("").str.count(",") >= 3
ext = ext[ok].copy()
ext["ingredient_count"] = ext.ingredients.str.count(",") + 1

mine = pd.read_csv("skincare_products.csv")

# NEW: Kaggle "Sephora Products and Skincare Reviews" (2023): download product_info.csv into data/raw/
# Its Sephora tags give acne-prone ('Good for: Acne/Blemishes') and specific skin types ('Best for Oily, Combo Skin').
import os, importlib, scrape_skincare as s
importlib.reload(s)
SEPHORA_2023 = "data/raw/product_info.csv"
parts = [mine]
if os.path.exists(SEPHORA_2023):
    parts.append(s.load_sephora_2023(SEPHORA_2023))
else:
    print("Not found (skipped):", SEPHORA_2023)
parts.append(ext)          # older Kaggle file last: newer Sephora data wins when the same product is in both

combined = pd.concat(parts, ignore_index=True)

# remove duplicates (same brand + name in both sources; keeps your scraped row)
key = combined.brand.str.lower().str.strip() + "|" + combined.name.str.lower().str.strip()
combined = combined[~key.duplicated(keep="first")]

combined.to_csv(OUT, index=False)
print(len(combined), "products;", combined.skin_types.notna().sum(), "with skin type")
print(combined.source.value_counts())

Kaggle columns: ['label', 'brand', 'name', 'price', 'rank', 'ingredients', 'combination', 'dry', 'normal', 'oily', 'sensitive']
Sephora 2023: 1951 skincare products with ingredients | 699 with skin type | 164 acne-prone
3299 products; 1717 with skin type
source
kaggle (sephora 2023)    1926
kaggle (sephora)         1068
brand site                224
shopify shop               81
Name: count, dtype: int64


## 4. Check the data (gate: 300+ products, 200+ with skin type)

In [7]:
import os
MERGED = "data/combined_products.csv" if os.path.exists("data/combined_products.csv") else "combined.csv"
df = pd.read_csv(MERGED)
print(len(df), "products;", df.skin_types.notna().sum(), "with skin type;", (df.ingredient_count > 0).sum(), "with ingredients")
print(df.brand.value_counts())
df.head()

3299 products; 1717 with skin type; 3178 with ingredients
brand
CLINIQUE              122
The Ordinary          111
SEPHORA COLLECTION    102
CeraVe                 68
Accufix                68
                     ... 
JOSIE MARAN             1
WANDER BEAUTY           1
EDIBLE BEAUTY           1
TOM FORD                1
BLACK UP                1
Name: count, Length: 228, dtype: int64


,brand,name,category,ingredients,ingredient_count,skin_types,helps_with,price_usd,rating,review_count,url,currency,source
0,The Ordinary,Multi-Peptide Lash and Brow Serum,Lash & Brow Treatments,"aqua/water/eau, glycerin, butylene glycol, myr...",32,all,Hair,17.90,NaN,NaN,https://theordinary.com/en-pk/multi-peptide-la...,USD,brand site
1,The Ordinary,Balancing & Clarifying Serum,Serums,"water, lactococcus ferment lysate, propanediol...",24,all,Signs of Congestion; Dryness; Textural Irregul...,23.90,NaN,NaN,https://theordinary.com/en-pk/balancing-clarif...,USD,brand site
2,The Ordinary,GF 15% Solution,Serums,"aqua/water/eau, butylene glycol, nicotiana ben...",12,all,Advanced Signs of Aging,18.60,NaN,NaN,https://theordinary.com/en-pk/gf-15-solution-g...,USD,brand site
3,The Ordinary,100% Cold-Pressed Virgin Marula Oil,Face Oils,NaN,0,all,Dryness; Antioxidant Support; Hair,13.70,NaN,NaN,https://theordinary.com/en-pk/100-cold-pressed...,USD,brand site
4,The Ordinary,Multi-Antioxidant Radiance Serum,Multi-Active,"aqua/water/eau, propanediol, ascorbyl glucosid...",21,all,Dullness; Antioxidant Support,16.80,NaN,NaN,https://theordinary.com/en-pk/multi-antioxidan...,USD,brand site


## 5. Find bad ingredient rows (marketing text instead of ingredients)

In [8]:
bad = df.ingredients.fillna("").str.split(",").apply(
    lambda p: len(p) > 0 and sum(len(x.split()) > 6 for x in p) > 0.15 * len(p))
print(bad.sum(), "suspicious rows")
df[bad][["brand", "name", "ingredient_count"]]

32 suspicious rows


,brand,name,ingredient_count
215,Cetaphil,Deep Cleansing Bar 3PK,6
312,Algenist,Gentle Rejuvenating Cleanser,30
424,caliray,Get Lit Skin and Mood Boosting Supplement,15
612,DAMDAM,Rice Drops Face Oil Hydrating Serum,10
912,Farmacy,Honey Grail Ultra-Hydrating Face Oil,5
1065,goop,G.Tox Detoxifying Superpowder,9
1124,HUM Nutrition,Collagen Love Skin Firming Supplement with Hya...,9
1125,HUM Nutrition,Calm Sweet Calm Stress Management Vegan Gummie...,12
1132,HUM Nutrition,OMG! Omega The Great Fish Oil Supplement,5
1267,Kiehl's Since 1851,Ultra Facial Moisturizing Cream with Squalane,30


In [9]:
# Optional: remove the bad rows, then run section 3 again to refetch them
# df[~bad].to_csv("skincare_products.csv", index=False)
import os
p = os.path.abspath("data/combined_filled.csv")
print(p, "->", "exists" if os.path.exists(p) else "NOT FOUND")

C:\Users\HP\Downloads\data\combined_filled.csv -> NOT FOUND


## 6. Fill helps_with and tag acne-prone
The next cell fills empty `helps_with` values, then adds `acne-prone` to products that are clearly for acne (name says acne/blemish/breakout/spot treatment, or benzoyl peroxide/adapalene). Borderline products are written to `acne_review_candidates.csv` for you to check.

In [10]:
import os, glob, importlib
import scrape_skincare as s
importlib.reload(s)          # picks up the new fill_helps_with / tag_acne_prone functions

print("Notebook folder:", os.getcwd())

# find the merged CSV (the one with the Kaggle rows in it)
candidates = glob.glob("**/combined*.csv", recursive=True)
print("Found:", candidates)

# use the file the Kaggle merge cell just wrote, so newly scraped brands are included
src = "data/combined_products.csv" if os.path.exists("data/combined_products.csv") \
      else next((c for c in candidates if "filled" not in c), None)
print("Using:", src)
if src is None:
    print("No combined CSV found. Run the Kaggle merge cell first.")
else:
    out = os.path.abspath("combined_filled.csv")
    s.fill_helps_with(src, out)
    s.tag_acne_prone(out, out, review_out="acne_review_candidates.csv")
    print("Saved to:", out, "| exists:", os.path.exists(out))

Notebook folder: C:\Users\HP\Downloads
Found: ['combined.csv', 'combined_filled.csv', 'data\\combined_products.csv', 'data\\raw\\combined.csv']
Using: data/combined_products.csv
helps_with filled: 1717 of 1797 | still empty: 80
  using your earlier yes/no answers from acne_review_candidates.csv
  of the clear ones: 117 from an oily-skin name (clarifying, oil-free, mattifying, pore) + pores tag or oil-control ingredient
acne-prone: tagged 175 clear + 85 from your review -> 465 total | 0 left to review in acne_review_candidates.csv
Saved to: C:\Users\HP\Downloads\combined_filled.csv | exists: True


In [11]:
# After you fill the 'add_acne_prone (yes/no)' column in acne_review_candidates.csv:
# rows marked yes get tagged, rows marked no are dropped from the list, blank rows stay for later.
# Safe to run again.
importlib.reload(s)
s.tag_acne_prone("combined_filled.csv", "combined_filled.csv",
                 review_out="acne_review_candidates.csv",
                 decisions="acne_review_candidates.csv")

  of the clear ones: 0 from an oily-skin name (clarifying, oil-free, mattifying, pore) + pores tag or oil-control ingredient
acne-prone: tagged 0 clear + 0 from your review -> 465 total | 303 left to review in acne_review_candidates.csv


,brand,name,category,ingredients,ingredient_count,skin_types,helps_with,price_usd,rating,review_count,url,currency,source,helps_with_source
0,The Ordinary,Multi-Peptide Lash and Brow Serum,Lash & Brow Treatments,"aqua/water/eau, glycerin, butylene glycol, myr...",32,all,Hair,17.90,NaN,NaN,https://theordinary.com/en-pk/multi-peptide-la...,USD,brand site,site
1,The Ordinary,Balancing & Clarifying Serum,Serums,"water, lactococcus ferment lysate, propanediol...",24,all|acne-prone,Signs of Congestion; Dryness; Textural Irregul...,23.90,NaN,NaN,https://theordinary.com/en-pk/balancing-clarif...,USD,brand site,site
2,The Ordinary,GF 15% Solution,Serums,"aqua/water/eau, butylene glycol, nicotiana ben...",12,all,Advanced Signs of Aging,18.60,NaN,NaN,https://theordinary.com/en-pk/gf-15-solution-g...,USD,brand site,site
3,The Ordinary,100% Cold-Pressed Virgin Marula Oil,Face Oils,NaN,0,all,Dryness; Antioxidant Support; Hair,13.70,NaN,NaN,https://theordinary.com/en-pk/100-cold-pressed...,USD,brand site,site
4,The Ordinary,Multi-Antioxidant Radiance Serum,Multi-Active,"aqua/water/eau, propanediol, ascorbyl glucosid...",21,all,Dullness; Antioxidant Support,16.80,NaN,NaN,https://theordinary.com/en-pk/multi-antioxidan...,USD,brand site,site
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3294,KATE SOMERVILLE,Daily Deflector™ Moisturizer Broad Spectrum SP...,Sun protect,"water, butylene glycol, peg-8, glycerin, undec...",41,NaN,Sun Protection; Dryness; Signs of Aging; Dehyd...,48,3.9,NaN,NaN,USD,kaggle (sephora),derived
3295,KORRES,Yoghurt Nourishing Fluid Veil Face Sunscreen B...,Sun protect,"water, alcohol denat., potassium cetyl phospha...",26,combination|dry|normal|oily|sensitive,Sun Protection; Dryness,35,3.9,NaN,NaN,USD,kaggle (sephora),derived
3296,KATE SOMERVILLE,Daily Deflector™ Waterlight Broad Spectrum SPF...,Sun protect,"water, isododecane, dimethicone, butyloctyl sa...",40,NaN,Sun Protection; Signs of Aging; Dehydration,48,3.6,NaN,NaN,USD,kaggle (sephora),derived
3297,VITA LIBERATA,Self Tan Dry Oil SPF 50,Sun protect,"water, dihydroxyacetone, glycerin, sclerocarya...",19,NaN,Sun Protection,54,3.5,NaN,NaN,USD,kaggle (sephora),derived
